# SparkOCR-VLM — Evaluation

Scores the OCR output in `workspace.default.ocr_silver` against committed ground-truth goldens.
Logs metrics to MLflow. Run this after `02_databricks_free`.

In [ ]:
print('Spark:', spark.version)

In [ ]:
%pip install -q /Volumes/workspace/default/sparkocr_demo/sparkocr_vlm-0.1.0-py3-none-any.whl
dbutils.library.restartPython()

In [ ]:
UC_TABLE = 'workspace.default.ocr_silver'

df = spark.table(UC_TABLE).toPandas()
print(f'Loaded {len(df)} rows from {UC_TABLE}')
display(df[['filename', 'page_num', 'markdown', 'cost_usd', 'error']])

In [ ]:
# Ground-truth goldens — inlined so no local filesystem dependency on Databricks
GOLDENS = {
    'synth_invoice': """# Invoice INV-2024-001

Bill to: ACME Corp
Date: 2024-01-15

| Item | Qty | Price | Total |
|---|---|---|---|
| Widget A | 10 | $25.00 | $250.00 |
| Widget B | 5 | $50.00 | $250.00 |
| Service Fee | 1 | $734.56 | $734.56 |

**Total: $1,234.56**""",

    'synth_report': """# Q1 2025 Quarterly Report

Prepared by: Finance Team

## Executive Summary

Revenue grew 18% year over year, driven by enterprise contracts. Operating margin improved to 22.4%.

## Detailed Results

- Revenue: $42.1M
- Gross margin: 71%
- Net income: $9.4M
- Headcount: 312
- Key risks: foreign exchange, supplier consolidation.""",

    'synth_table': """# Sales by Region

| Region | Q1 | Q2 | Q3 |
|---|---|---|---|
| North | 100 | 120 | 140 |
| South | 80 | 90 | 110 |
| East | 60 | 70 | 85 |
| West | 150 | 160 | 175 |""",
}

ANCHORS = {
    'synth_invoice': ['INV-2024-001', '$1,234.56', 'ACME Corp'],
    'synth_report':  ['42.1M', '22.4%', 'Finance Team'],
    'synth_table':   ['North', 'South', 'East', 'West'],
}

print('Goldens loaded for:', list(GOLDENS.keys()))

In [ ]:
from pathlib import Path
from sparkocr_vlm.evaluator import (
    normalized_edit_distance,
    anchor_recall,
    table_cell_f1,
    reading_order_edit_distance,
)

results = []
for _, row in df.iterrows():
    stem = Path(str(row['filename'])).stem
    gt = GOLDENS.get(stem)
    actual = str(row['markdown'] or '')
    if gt is None:
        continue
    results.append({
        'file':       row['filename'],
        'page':       row['page_num'],
        'edit_dist':  round(normalized_edit_distance(actual, gt), 4),
        'anchor_rec': round(anchor_recall(actual, ANCHORS.get(stem, [])), 4),
        'table_f1':   round(table_cell_f1(actual, gt), 4),
        'reading_ed': round(reading_order_edit_distance(actual, gt), 4),
    })

import pandas as pd
scores = pd.DataFrame(results)
display(scores)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

metrics = ['edit_dist', 'anchor_rec', 'table_f1', 'reading_ed']
labels  = ['Edit Distance↓', 'Anchor Recall↑', 'Table F1↑', 'Reading Order ED↓']

x = np.arange(len(scores))
width = 0.2

fig, ax = plt.subplots(figsize=(10, 4))
for i, (m, l) in enumerate(zip(metrics, labels)):
    ax.bar(x + i * width, scores[m], width, label=l)

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels([f"{r['file']}\np{r['page']}" for _, r in scores.iterrows()], fontsize=8)
ax.set_ylim(0, 1.05)
ax.set_title('SparkOCR-VLM — per-page eval metrics')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
import mlflow

means = scores[['edit_dist','anchor_rec','table_f1','reading_ed']].mean().round(4)
print('=== Aggregate metrics ===')
for k, v in means.items():
    print(f'  {k}: {v}')

with mlflow.start_run(run_name='sparkocr-eval'):
    for k, v in means.items():
        mlflow.log_metric(k, float(v))
    mlflow.log_metric('n_pages_scored', len(scores))
    print('\nLogged to MLflow.')